# IA1 - Analyse individuelle AQI

Ce notebook analyse le fichier `data/clean/clean.csv` produit par le pipeline DONNEES2 de SquadAnalytics. L'objectif est de preparer les visualisations et les insights presentes dans le dashboard IA1.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "clean" / "clean.csv").exists():
    ROOT = ROOT.parent
CLEAN_CSV = ROOT / "data" / "clean" / "clean.csv"
WAREHOUSE = ROOT / "data" / "warehouse.db"

df = pd.read_csv(CLEAN_CSV, parse_dates=["datetime"])
df = df.sort_values(["datetime", "ville"]).reset_index(drop=True)
df.head()


In [ ]:
print("Nombre de lignes:", len(df))
print("Villes:", sorted(df["ville"].unique()))
print("Periode:", df["datetime"].min(), "->", df["datetime"].max())
print("Doublons ville+heure:", df.duplicated(["ville", "datetime"]).sum())

df.isna().sum()


In [ ]:
city_summary = (
    df.groupby("ville")
    .agg(
        lignes=("aqi", "size"),
        aqi_moyen=("aqi", "mean"),
        aqi_max=("aqi", "max"),
        pm10_moyen=("pm10", "mean"),
        pm25_moyen=("pm2_5", "mean"),
        co_moyen=("co", "mean"),
        no2_moyen=("no2", "mean"),
    )
    .round(2)
    .sort_values("aqi_moyen", ascending=False)
)
city_summary


In [ ]:
daily_aqi = (
    df.assign(date=df["datetime"].dt.date)
    .groupby(["date", "ville"], as_index=False)["aqi"]
    .mean()
)

ax = daily_aqi.pivot(index="date", columns="ville", values="aqi").plot(
    figsize=(12, 5),
    title="Evolution quotidienne de l'AQI moyen par ville",
)
ax.set_xlabel("Date")
ax.set_ylabel("AQI moyen")


In [ ]:
hourly_profile = (
    df.assign(heure=df["datetime"].dt.hour)
    .groupby(["heure", "ville"], as_index=False)["aqi"]
    .mean()
)

ax = hourly_profile.pivot(index="heure", columns="ville", values="aqi").plot(
    figsize=(12, 5),
    title="Profil horaire moyen de l'AQI",
)
ax.set_xlabel("Heure")
ax.set_ylabel("AQI moyen")


In [ ]:
conn = sqlite3.connect(WAREHOUSE)
query = '''
SELECT f.ville, ROUND(AVG(f.aqi), 2) AS aqi_moyen, MAX(f.aqi) AS aqi_max, COUNT(*) AS lignes
FROM fact_qualite_air f
GROUP BY f.ville
ORDER BY aqi_moyen DESC;
'''
pd.read_sql_query(query, conn)


## Insights principaux
- Dakar affiche l'AQI moyen le plus eleve (82.03).
- Antananarivo est la ville la plus favorable sur la periode (24.59 en moyenne).
- Le pic observe atteint 181.0 a Dakar le 2026-05-17T09:00.
- Le fichier clean respecte le contrat IA1/DONNEES2: une ligne par ville et par heure, sans doublons.